# DrivingCopilotAgent — 로컬 모델 서버 (Colab GPU)

이 노트북은 `app/model_server`(Ollama를 대체하는 OpenAI 호환 추론 서버)를
Google Colab의 GPU에서 실행한다. 로컬 머신 GPU가 8GB(RTX 4060 Ti)로 두 모델
(Qwen2-VL-7B-Instruct(bnb 4bit) + Qwen2.5-1.5B-Instruct)을 동시에 올리기 타이트할 때,
Colab의 더 큰 GPU(T4 16GB 이상)를 빌려 쓰기 위한 용도다.

**사용 순서**
1. 상단 메뉴 `런타임 > 런타임 유형 변경`에서 GPU(T4 이상)를 선택한다.
2. 아래 셀을 위에서 아래로 순서대로 실행한다.
3. ngrok으로 발급된 공개 URL을 로컬 레포의 `.env`의 `MODEL_SERVER_URL`에 넣고
   `MODEL_SERVER_URL=<ngrok-url>/v1` 로 저장한다.
4. 로컬에서 `bash scripts/run_agents.sh`를 실행하면 4개 에이전트 프로세스가 이
   Colab 서버로 추론 요청을 보낸다.

**주의**
- ngrok 무료 티어 URL은 이 노트북을 다시 실행할 때마다 바뀐다 — 재실행 시
  `.env`도 다시 갱신해야 한다.
- Colab 세션은 유휴 시간이 길어지면 끊긴다 — 상시 운영용이 아니라 테스트/개발용이다.
- VL 모델은 bitsandbytes 4bit로 로딩 시점에 즉석 양자화한다(GPTQ+Marlin 커널
  JIT 컴파일이 Colab 무료 T4에서 멈추는 문제가 있어 피했다) — 대신 비양자화
  체크포인트(~15GB)를 받아야 해서 최초 다운로드가 더 오래 걸릴 수 있다.


## 0. GPU 확인

In [ ]:
!nvidia-smi


## 1. 의존성 설치

Colab은 CUDA에 맞춰 빌드된 `torch`를 이미 갖고 있으므로 재설치하지 않는다 —
재설치하면 오히려 CUDA 버전이 안 맞아 깨질 수 있다.


In [ ]:
%pip install -q transformers accelerate bitsandbytes qwen-vl-utils fastapi "uvicorn[standard]" pyngrok requests


## 2. 모델 서버 코드 작성

`app/model_server/backend.py` + `server.py`와 동일한 로직을 이 노트북 하나로
재현한다 — 전체 레포를 클론하지 않고도 돌아가도록 `app.core.config`/
`app.core.json_utils` 의존성을 인라인했다. 실제 레포와 동작을 맞추려면
로컬의 `app/model_server/`와 이 파일을 함께 갱신할 것.


In [ ]:
%%writefile model_server.py
"""
Colab 전용 독립 실행 버전 — DrivingCopilotAgent의 app/model_server를
레포 클론 없이 이 노트북 하나로 재현한 것. 로직은 app/model_server/backend.py +
server.py와 동일하며, app.core.config/app.core.json_utils 의존성만 인라인했다.
"""

from __future__ import annotations

import json
import logging
import os
import re
import threading
import time
import uuid
from contextlib import asynccontextmanager
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse
from starlette.requests import Request

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("model_server")

QWEN_VL_MODEL_NAME = os.getenv("QWEN_VL_MODEL_NAME", "Qwen/Qwen2-VL-7B-Instruct")
QWEN_TEXT_MODEL_NAME = os.getenv("QWEN_TEXT_MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
PORT = int(os.getenv("MODEL_SERVER_PORT", "11500"))


def extract_first_json_object(text: str) -> str:
    """텍스트에서 첫 번째로 완성되는 최상위 JSON 객체만 잘라서 반환한다."""
    start = text.find("{")
    if start == -1:
        return text
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    return text[start:]


def _resolve_dtype():
    import torch
    return torch.float16 if torch.cuda.is_available() else torch.float32


@dataclass
class LoadedTextModel:
    tokenizer: Any
    model: Any

    def _build_inputs(self, messages, tools):
        text = self.tokenizer.apply_chat_template(
            messages, tools=tools or None, tokenize=False, add_generation_prompt=True,
        )
        return self.tokenizer([text], return_tensors="pt").to(self.model.device)

    def generate(self, messages, tools=None, max_new_tokens=512, temperature=0.0):
        import torch
        inputs = self._build_inputs(messages, tools)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0.0,
                temperature=temperature if temperature > 0.0 else None,
            )
        generated = output_ids[0][inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(generated, skip_special_tokens=True).strip()

    def stream_generate(self, messages, max_new_tokens=512, temperature=0.0):
        from transformers import TextIteratorStreamer
        inputs = self._build_inputs(messages, tools=None)
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True)
        kwargs = dict(
            **inputs, max_new_tokens=max_new_tokens, do_sample=temperature > 0.0,
            temperature=temperature if temperature > 0.0 else None, streamer=streamer,
        )
        thread = threading.Thread(target=self.model.generate, kwargs=kwargs)
        thread.start()
        for token_text in streamer:
            if token_text:
                yield token_text
        thread.join()


@dataclass
class LoadedVLModel:
    processor: Any
    model: Any

    def _build_inputs(self, messages):
        from qwen_vl_utils import process_vision_info
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        return self.processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt",
        ).to(self.model.device)

    def generate(self, messages, max_new_tokens=512, temperature=0.0):
        import torch
        inputs = self._build_inputs(messages)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0.0,
                temperature=temperature if temperature > 0.0 else None,
            )
        generated = output_ids[:, inputs["input_ids"].shape[-1]:]
        return self.processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

    def stream_generate(self, messages, max_new_tokens=512, temperature=0.0):
        from transformers import TextIteratorStreamer
        inputs = self._build_inputs(messages)
        streamer = TextIteratorStreamer(
            self.processor.tokenizer, skip_prompt=True, skip_special_tokens=True
        )
        kwargs = dict(
            **inputs, max_new_tokens=max_new_tokens, do_sample=temperature > 0.0,
            temperature=temperature if temperature > 0.0 else None, streamer=streamer,
        )
        thread = threading.Thread(target=self.model.generate, kwargs=kwargs)
        thread.start()
        for token_text in streamer:
            if token_text:
                yield token_text
        thread.join()


_text_model: Optional[LoadedTextModel] = None
_vl_model: Optional[LoadedVLModel] = None


def load_models() -> None:
    global _text_model, _vl_model
    if _text_model is None:
        logger.info("텍스트 모델 로딩 시작: %s", QWEN_TEXT_MODEL_NAME)
        from transformers import AutoModelForCausalLM, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(QWEN_TEXT_MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(
            QWEN_TEXT_MODEL_NAME, device_map="auto", torch_dtype=_resolve_dtype(),
        )
        _text_model = LoadedTextModel(tokenizer=tokenizer, model=model)
        logger.info("텍스트 모델 로딩 완료")

    if _vl_model is None:
        logger.info("VL 모델 로딩 시작 (1/3: processor): %s", QWEN_VL_MODEL_NAME)
        from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration
        processor = AutoProcessor.from_pretrained(QWEN_VL_MODEL_NAME)
        logger.info("VL 모델 로딩 (2/3: 가중치 다운로드+GPU 디스패치, bnb 4bit 즉석 양자화)")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=_resolve_dtype(), bnb_4bit_quant_type="nf4",
        )
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            QWEN_VL_MODEL_NAME, device_map="auto", quantization_config=quantization_config,
        )
        logger.info("VL 모델 로딩 (3/3: from_pretrained 반환됨, 객체 구성 중)")
        _vl_model = LoadedVLModel(processor=processor, model=model)
        logger.info("VL 모델 로딩 완료")


def is_ready() -> bool:
    return _text_model is not None and _vl_model is not None


@asynccontextmanager
async def lifespan(app: FastAPI):
    threading.Thread(target=load_models, daemon=True).start()
    yield


app = FastAPI(title="Local Qwen Model Server (Colab, OpenAI-compatible)", lifespan=lifespan)


@app.get("/health")
async def health():
    return {"status": "ok" if is_ready() else "loading"}


_TOOL_CALL_RE = re.compile(r"<tool_call>(.*?)</tool_call>", re.DOTALL)


def _normalize_content_for_vl(content: Any) -> Any:
    """tool_calls를 담은 assistant 메시지는 content=None으로 오므로 빈 문자열로
    취급한다 — Qwen2-VL 채팅 템플릿은 content가 문자열이 아니면 무조건 순회
    가능한 블록 리스트로 가정하고 for-loop을 돌리므로, None을 그대로 넘기면
    'NoneType' object is not iterable 로 죽는다(_normalize_content_for_text와
    동일한 이유 — supervisor는 항상 QWEN_VL_MODEL_NAME을 쓰고 대화 히스토리
    전체(messages_to_send.extend(messages))를 함께 보내므로, knowledge의
    ReAct 루프가 state["messages"]에 남긴 tool_calls 메시지가 여기로도 들어온다)."""
    if content is None:
        return ""
    if not isinstance(content, list):
        return content
    normalized = []
    for block in content:
        if block.get("type") == "image_url":
            normalized.append({"type": "image", "image": block["image_url"]["url"]})
        else:
            normalized.append(block)
    return normalized


def _normalize_content_for_text(content: Any) -> str:
    """content가 블록 리스트면 text 블록만 이어붙인다. tool_calls를 담은
    assistant 메시지는 content=None으로 오므로 빈 문자열로 취급한다
    (str(None) == "None" 문자열이 그대로 들어가는 사고 방지)."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(b.get("text", "") for b in content if b.get("type") == "text")
    return str(content)


def _normalize_tool_call_arguments(tool_call: Dict[str, Any]) -> Dict[str, Any]:
    """OpenAI 왕복 규약은 function.arguments를 JSON 문자열로 담는다(_parse_tool_calls
    참고). Qwen 채팅 템플릿은 `tool_call.arguments | tojson`으로 객체를 렌더링하므로,
    문자열을 그대로 넘기면 따옴표로 한 번 더 감싸져(이중 인코딩) 모델이 자신이
    과거에 만든 tool_call을 스스로 알아보지 못한다 — 여기서 dict로 되돌린다."""
    function = dict(tool_call.get("function") or {})
    raw_args = function.get("arguments", "{}")
    if isinstance(raw_args, str):
        try:
            function["arguments"] = json.loads(raw_args) if raw_args else {}
        except json.JSONDecodeError:
            function["arguments"] = {}
    return {**tool_call, "function": function}


def _normalize_message_for_text(message: Dict[str, Any]) -> Dict[str, Any]:
    """OpenAI 포맷 메시지를 텍스트 모델의 Qwen 채팅 템플릿이 기대하는 형태로
    정규화한다. role/content만 남기고 재조립하면 assistant의 tool_calls와
    tool 메시지가 사라져, ReAct 에이전트(knowledge_node)가 자신이 이미 tool을
    호출·수신했다는 사실을 다음 턴에서 볼 수 없다 — 매 턴 같은 tool을 다시
    호출하며 recursion_limit까지 수렴하지 못하는 원인이었다.
    - assistant + tool_calls: content(falsy 허용) 그대로 유지, tool_calls는
      arguments를 dict로 되돌려 템플릿의 이중 인코딩을 막는다.
    - tool: content만 전달한다 — 템플릿은 tool_call_id/name을 쓰지 않고,
      인접한 tool 메시지들을 role만으로 <tool_response> 블록에 함께 묶는다.
    """
    role = message.get("role", "user")
    tool_calls = message.get("tool_calls")

    if role == "assistant" and tool_calls:
        return {
            "role": "assistant",
            "content": _normalize_content_for_text(message.get("content")),
            "tool_calls": [_normalize_tool_call_arguments(tc) for tc in tool_calls],
        }
    return {"role": role, "content": _normalize_content_for_text(message.get("content"))}


def _has_image(messages: List[Dict[str, Any]]) -> bool:
    return any(
        isinstance(m.get("content"), list)
        and any(b.get("type") == "image_url" for b in m["content"])
        for m in messages
    )


def _response_format_instruction(response_format: Optional[Dict[str, Any]]) -> Optional[str]:
    if not response_format:
        return None
    fmt_type = response_format.get("type")
    if fmt_type == "json_schema":
        schema = response_format.get("json_schema", {}).get("schema", {})
        return (
            "You MUST respond with ONLY a single JSON object that strictly matches "
            f"this JSON Schema, with no prose or markdown fences:\n{json.dumps(schema)}"
        )
    if fmt_type == "json_object":
        return "You MUST respond with ONLY a single valid JSON object, no prose or markdown fences."
    return None


def _apply_response_format(messages, response_format):
    instruction = _response_format_instruction(response_format)
    if not instruction:
        return messages
    return [*messages, {"role": "system", "content": instruction}]


def _parse_tool_calls(raw_text: str):
    matches = _TOOL_CALL_RE.findall(raw_text)
    if not matches:
        return None
    tool_calls = []
    for block in matches:
        try:
            parsed = json.loads(extract_first_json_object(block.strip()))
        except (json.JSONDecodeError, AttributeError):
            continue
        tool_calls.append({
            "id": f"call_{uuid.uuid4().hex[:24]}",
            "type": "function",
            "function": {
                "name": parsed.get("name", ""),
                "arguments": json.dumps(parsed.get("arguments", {})),
            },
        })
    return tool_calls or None


def _strip_tool_call_blocks(raw_text: str) -> str:
    return _TOOL_CALL_RE.sub("", raw_text).strip()


def _completion_payload(model: str, content, tool_calls):
    message: Dict[str, Any] = {"role": "assistant", "content": content}
    finish_reason = "stop"
    if tool_calls:
        message["tool_calls"] = tool_calls
        message["content"] = None
        finish_reason = "tool_calls"
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:24]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": model,
        "choices": [{"index": 0, "message": message, "finish_reason": finish_reason}],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    }


def _stream_chunk(model: str, delta, finish_reason=None):
    payload = {
        "id": f"chatcmpl-{uuid.uuid4().hex[:24]}",
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": model,
        "choices": [{"index": 0, "delta": delta, "finish_reason": finish_reason}],
    }
    return f"data: {json.dumps(payload)}\n\n"


@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    model_name: str = body.get("model", "")
    messages: List[Dict[str, Any]] = body.get("messages", [])
    temperature: float = float(body.get("temperature") or 0.0)
    max_tokens: int = int(body.get("max_tokens") or 512)
    stream: bool = bool(body.get("stream", False))
    response_format = body.get("response_format")
    tools = body.get("tools")

    is_vl = model_name == QWEN_VL_MODEL_NAME or _has_image(messages)

    if is_vl:
        if _vl_model is None:
            return JSONResponse({"error": "VL model not loaded yet, check /health"}, status_code=503)
        vl_messages = _apply_response_format(
            [{"role": m.get("role", "user"), "content": _normalize_content_for_vl(m.get("content"))}
             for m in messages],
            response_format,
        )
        if stream:
            async def _gen():
                for token in _vl_model.stream_generate(vl_messages, max_new_tokens=max_tokens, temperature=temperature):
                    yield _stream_chunk(model_name, {"content": token})
                yield _stream_chunk(model_name, {}, finish_reason="stop")
                yield "data: [DONE]\n\n"
            return StreamingResponse(_gen(), media_type="text/event-stream")

        raw = _vl_model.generate(vl_messages, max_new_tokens=max_tokens, temperature=temperature)
        return JSONResponse(_completion_payload(model_name, raw, None))

    if _text_model is None:
        return JSONResponse({"error": "text model not loaded yet, check /health"}, status_code=503)

    text_messages = _apply_response_format(
        [_normalize_message_for_text(m) for m in messages],
        response_format,
    )

    if stream:
        async def _gen():
            for token in _text_model.stream_generate(text_messages, max_new_tokens=max_tokens, temperature=temperature):
                yield _stream_chunk(model_name, {"content": token})
            yield _stream_chunk(model_name, {}, finish_reason="stop")
            yield "data: [DONE]\n\n"
        return StreamingResponse(_gen(), media_type="text/event-stream")

    raw = _text_model.generate(text_messages, tools=tools, max_new_tokens=max_tokens, temperature=temperature)
    tool_calls = _parse_tool_calls(raw) if tools else None
    content = None if tool_calls else _strip_tool_call_blocks(raw)
    return JSONResponse(_completion_payload(model_name, content, tool_calls))


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=PORT)

## 3. ngrok 인증 토큰

무료 계정으로 발급받은 authtoken이 필요하다 —
https://dashboard.ngrok.com/get-started/your-authtoken 에서 확인.
(입력값은 화면에 노출되지 않는다.)


In [ ]:
import getpass
from pyngrok import ngrok

ngrok.set_auth_token(getpass.getpass("ngrok authtoken: "))


## 4. 모델 서버 기동 + 공개 URL 발급

최초 실행 시 HuggingFace에서 7B(비양자화, ~15GB) + 1.5B 가중치를 내려받는다 —
네트워크 상태에 따라 꽤 걸릴 수 있다. `/health`가 `"ok"`가 될 때까지
자동으로 기다린다.


In [ ]:
import os
import subprocess
import time

import requests
from pyngrok import ngrok

# 필요하면 다른 모델로 바꿀 수 있다 (예: VRAM이 더 타이트하면 더 작은 양자화 모델).
# os.environ["QWEN_VL_MODEL_NAME"] = "Qwen/Qwen2-VL-7B-Instruct"
# os.environ["QWEN_TEXT_MODEL_NAME"] = "Qwen/Qwen2.5-1.5B-Instruct"

server_proc = subprocess.Popen(
    ["python", "model_server.py"],
    stdout=open("server.log", "w"),
    stderr=subprocess.STDOUT,
    # 노트북 셀과 별도 프로세스 그룹으로 띄운다 — 안 그러면 Colab "실행 중단"이
    # 셀뿐 아니라 이 서버 프로세스(uvicorn)에도 SIGINT를 보내 같이 꺼진다.
    start_new_session=True,
)

tunnel = ngrok.connect(11500, "http")
public_url = tunnel.public_url
print("공개 URL:", public_url)
print("로컬 .env에 다음 줄을 추가/갱신하세요:")
print(f"MODEL_SERVER_URL={public_url}/v1")

print("\n모델 로딩 대기 중...")
while True:
    try:
        r = requests.get("http://localhost:11500/health", timeout=3)
        if r.json().get("status") == "ok":
            print("모델 서버 준비 완료.")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(5)


## 5. 동작 확인

1.5B 텍스트 모델로 간단한 질문을 보내본다. 정상이면 `choices[0].message.content`에
답변이 담겨 온다.


In [ ]:
import requests

resp = requests.post(
    f"{public_url}/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "messages": [{"role": "user", "content": "안녕! 한 문장으로 자기소개 해줘."}],
        "temperature": 0.3,
        "max_tokens": 100,
    },
    timeout=120,
)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])


## 6. (문제 발생 시) 서버 로그 확인

`/health`가 계속 `"loading"`에 머물거나 요청이 실패하면 로그를 먼저 확인한다.


In [ ]:
!tail -n 80 server.log


## 7. 종료

노트북을 다 쓰고 나면 서버 프로세스와 터널을 정리한다 (런타임을 끄면 자동으로
정리되지만, 같은 세션에서 재시작하려면 먼저 아래를 실행).


In [ ]:
ngrok.disconnect(tunnel.public_url)
server_proc.terminate()
